In [1]:
# Please execute/shift-return this cell everytime you run the notebook.  Don't edit it. 
%load_ext autoreload
%autoreload 2
from notebook import * 

### Matrix tiling algorithm

Let's try to partition GEMM into smaller tiles!

In [2]:
render_code("matrix_mul/blockmm.c", show=["//START","//END"])

// matrix_mul/blockmm.c:59-77 (19 lines)
//START
void blockmm(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  uint64_t i,j,k, ii, jj, kk;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {        
          for(ii = i; ii < i+tile_size; ii++)
            for(jj = j; jj < j+tile_size; jj++)
              for(kk = k; kk < k+tile_size; kk++)
                c[ii][jj] += a[ii][kk]*b[kk][jj];
      }
    }
  }  
}
//END

In [3]:
! cd matrix_mul/; make clean blockmm

rm -f blockmm mm blockmm_transpose cachegrind.* mm_dump rect_blockmm_trans blockmm_transpose_reg blockmm_reg
gcc -O4 -DHAVE_LINUX_PERF_EVENT_H blockmm.c perfstats.c -o blockmm 
blockmm.c: In function ‘main’:
blockmm.c:48:16: warning: format ‘%lu’ expects argument of type ‘long unsigned int’, but argument 3 has type ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wformat=-Wformat=]8;;]
   48 |   printf("%d,%lu,",ARRAY_SIZE,tile_size);
      |              ~~^              ~~~~~~~~~
      |                |              |
      |                |              int
      |                long unsigned int
      |              %u


## Try with tile size == 32

In [4]:
! cd matrix_mul; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm.csv
! ./matrix_mul/blockmm 512 32 >> ./matrix_mul/blockmm.csv ;./matrix_mul/blockmm 1024 32 >> ./matrix_mul/blockmm.csv ; ./matrix_mul/blockmm 2048 32 >> ./matrix_mul/blockmm.csv; ./matrix_mul/blockmm 4096 32 >> ./matrix_mul/blockmm.csv

In [5]:
display_df_mono(render_csv("matrix_mul/mm.csv"))
display_df_mono(render_csv("matrix_mul/blockmm.csv"))

,index,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,512,1062500352,667529356,0.628263,0.195576,0.130553,0.251272,133247614,530292258
1,1024,8565406019,10432234118,1.217950,0.234545,2.446827,0.232808,995509066,4276088710
2,2048,69009270640,122219488197,1.771059,0.192883,23.574024,0.314315,10830867662,34458636578


,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,512,32,1117804933,392646546,0.351266,0.192713,0.075668,0.214554,118048123,550202486
1,1,1024,32,8730124105,3121765059,0.357585,0.201141,0.627914,0.206149,885844987,4297111314
2,2,2048,32,71543096326,27616887651,0.386018,0.192817,5.325003,0.217683,7665523622,35214204432
3,3,4096,32,572327950562,229501971586,0.400997,0.192879,44.266209,0.222544,62691514828,281703756231


## Try with tile size == 8

In [6]:
! cd matrix_mul; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm.csv
! ./matrix_mul/blockmm 512 8 >> ./matrix_mul/blockmm.csv ;./matrix_mul/blockmm 1024 8 >> ./matrix_mul/blockmm.csv ; ./matrix_mul/blockmm 2048 8 >> ./matrix_mul/blockmm.csv; ./matrix_mul/blockmm 4096 8 >> ./matrix_mul/blockmm.csv

In [7]:
display_df_mono(render_csv("matrix_mul/mm.csv"))
display_df_mono(render_csv("matrix_mul/blockmm.csv"))

,index,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,512,1062500352,667529356,0.628263,0.195576,0.130553,0.251272,133247614,530292258
1,1024,8565406019,10432234118,1.217950,0.234545,2.446827,0.232808,995509066,4276088710
2,2048,69009270640,122219488197,1.771059,0.192883,23.574024,0.314315,10830867662,34458636578


,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,512,8,1256611517,258013485,0.205325,0.196986,0.050825,0.008987,5316313,591558610
1,1,1024,8,10129812092,2172909231,0.214506,0.192991,0.419351,0.012879,61416597,4768601485
2,2,2048,8,81046054519,20961741053,0.258640,0.192839,4.042239,0.010700,408203923,38151543454
3,3,4096,8,648413773151,192372302456,0.296681,0.192841,37.097186,0.012444,3798364992,305228772767


In [8]:
! ./matrix_mul/blockmm 2048 4 >> ./matrix_mul/blockmm.csv
! ./matrix_mul/blockmm 2048 16 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 32 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 64 >> ./matrix_mul/blockmm.csv
! ./matrix_mul/blockmm 2048 128 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 256 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 512 >> ./matrix_mul/blockmm.csv 
display_df_mono(render_csv("matrix_mul/blockmm.csv"))

,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,512,8,1256611517,258013485,0.205325,0.196986,0.050825,0.008987,5316313,591558610
1,1,1024,8,10129812092,2172909231,0.214506,0.192991,0.419351,0.012879,61416597,4768601485
2,2,2048,8,81046054519,20961741053,0.258640,0.192839,4.042239,0.010700,408203923,38151543454
3,3,4096,8,648413773151,192372302456,0.296681,0.192841,37.097186,0.012444,3798364992,305228772767
4,4,2048,4,97770200805,26860512983,0.274731,0.193094,5.186593,0.015354,670113418,43643206828
5,5,2048,16,74472732908,19065453359,0.256006,0.192844,3.676662,0.071555,2583498815,36104917769
6,6,2048,32,71445026605,27586227133,0.386118,0.193274,5.331703,0.217336,7642831462,35165945185
7,7,2048,64,70149978739,32487713206,0.463118,0.192812,6.264029,0.242866,8450385350,34794390699
8,8,2048,128,69468060768,34473229060,0.496246,0.192907,6.650128,0.246043,8510511366,34589546912
9,9,2048,256,69130101329,35068937412,0.507289,0.192840,6.762704,0.247026,8519450536,34488096149


In [15]:
render_code("matrix_mul/blockmm_reg.c", show=["//START","//END"])

// matrix_mul/blockmm_reg.c:59-82 (24 lines)
//START
void blockmm(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  uint64_t i,j,k, ii, jj, kk;
    double result = 0;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {        
          for(ii = i; ii < i+tile_size; ii++)
            for(jj = j; jj < j+tile_size; jj++)
                {
                result = 0;
                for(kk = k; kk < k+tile_size; kk++)
                    result += a[ii][kk]*b[kk][jj];
                c[ii][jj] += result;
          }
      }
    }
  }  
}
//END

In [16]:
! cd matrix_mul; make blockmm_reg; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_reg.csv
! ./matrix_mul/blockmm_reg 2048 4 >> ./matrix_mul/blockmm_reg.csv
! ./matrix_mul/blockmm_reg 2048 8 >> ./matrix_mul/blockmm_reg.csv
! ./matrix_mul/blockmm_reg 2048 16 >> ./matrix_mul/blockmm_reg.csv 
! ./matrix_mul/blockmm_reg 2048 32 >> ./matrix_mul/blockmm_reg.csv 
! ./matrix_mul/blockmm_reg 2048 64 >> ./matrix_mul/blockmm_reg.csv
! ./matrix_mul/blockmm_reg 2048 128 >> ./matrix_mul/blockmm_reg.csv 
! ./matrix_mul/blockmm_reg 2048 256 >> ./matrix_mul/blockmm_reg.csv 
! ./matrix_mul/blockmm_reg 2048 512 >> ./matrix_mul/blockmm_reg.csv 
display_df_mono(render_csv("matrix_mul/blockmm_reg.csv"))

gcc -O4 -DHAVE_LINUX_PERF_EVENT_H blockmm_reg.c perfstats.c -o blockmm_reg 
blockmm_reg.c: In function ‘main’:
blockmm_reg.c:48:16: warning: format ‘%lu’ expects argument of type ‘long unsigned int’, but argument 3 has type ‘int’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wformat=-Wformat=]8;;]
   48 |   printf("%d,%lu,",ARRAY_SIZE,tile_size);
      |              ~~^              ~~~~~~~~~
      |                |              |
      |                |              int
      |                long unsigned int
      |              %u


,index,size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses
0,0,2048,4,106231150958,27252094899,0.256536,0.192838,5.255226,0.018459,778384508,42169347374
1,1,2048,8,78810054718,18649129501,0.236634,0.192875,3.596947,0.012925,397879053,30784852772
2,2,2048,16,66921747338,15443634623,0.230772,0.192836,2.978092,0.100651,2610613110,25937304214
3,3,2048,32,61315372824,18297510381,0.298416,0.192807,3.527895,0.296224,7012477421,23672910712
4,4,2048,64,58580600841,20738095573,0.354010,0.192903,4.000432,0.378599,8546354888,22573608147
5,5,2048,128,57224706635,20518268835,0.358556,0.192807,3.956075,0.394433,8689345489,22029980561
6,6,2048,256,56561303385,25760861157,0.455450,0.192862,4.968286,0.398078,8663766687,21763972970
7,7,2048,512,56275561344,49783249145,0.884634,0.192841,9.600270,0.399357,8645320536,21648112878


In [17]:
render_code("matrix_mul/blockmm_transpose.c", show=["//START","//END"])

// matrix_mul/blockmm_transpose.c:63-81 (19 lines)
//START
void blockmm_transpose(double **a, double **b, double **c, uint64_t M, uint64_t N, uint64_t K)
{
  int i,j,k, ii, jj, kk;
  for(i = 0; i < M; i+=tile_size)
  {
    for(j = 0; j < K; j+=tile_size)
    {
      for(k = 0; k < N; k+=tile_size)
      {        
          for(ii = i; ii < i+tile_size; ii++)
            for(jj = j; jj < j+tile_size; jj++)
              for(kk = k; kk < k+tile_size; kk++)
                c[ii][jj] += a[ii][kk]*b[jj][kk];
      }
    }
  }  
}
//END

### Matrix transpose

In [ ]:
! cd matrix_mul; rm blockmm_transpose; make blockmm_transpose; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_transpose.csv
! ./matrix_mul/blockmm_transpose 512 8 >> ./matrix_mul/blockmm_transpose.csv ;./matrix_mul/blockmm_transpose 1024 8 >> ./matrix_mul/blockmm_transpose.csv ; ./matrix_mul/blockmm_transpose 2048 8 >> ./matrix_mul/blockmm_transpose.csv; ./matrix_mul/blockmm_transpose 4096 8 >> ./matrix_mul/blockmm_transpose.csv

rm: cannot remove 'blockmm_transpose': No such file or directory
gcc -O4 -DHAVE_LINUX_PERF_EVENT_H blockmm_transpose.c perfstats.c -o blockmm_transpose
234410496.000000,1406510080.000000,10521102336.000000,

In [ ]:
! ./matrix_mul/blockmm_transpose 2048 8 >> ./matrix_mul/blockmm_transpose.csv 
! ./matrix_mul/blockmm_transpose 2048 16 >> ./matrix_mul/blockmm_transpose.csv 
! ./matrix_mul/blockmm_transpose 2048 32 >> ./matrix_mul/blockmm_transpose.csv 
! ./matrix_mul/blockmm_transpose 2048 64 >> ./matrix_mul/blockmm_transpose.csv
! ./matrix_mul/blockmm_transpose 2048 128 >> ./matrix_mul/blockmm_transpose.csv
! ./matrix_mul/blockmm_transpose 2048 256 >> ./matrix_mul/blockmm_transpose.csv

In [ ]:
display_df_mono(render_csv("matrix_mul/blockmm_transpose.csv"))

In [ ]:
display_df_mono(render_csv("matrix_mul/blockmm.csv"))

In [ ]:
render_code("matrix_mul/blockmm_transpose_reg.c", show=["//START","//END"])

In [ ]:
! cd matrix_mul; rm blockmm_transpose_reg; make blockmm_transpose_reg; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_transpose_reg.csv 
! ./matrix_mul/blockmm_transpose_reg 2048 8 >> ./matrix_mul/blockmm_transpose_reg.csv 
! ./matrix_mul/blockmm_transpose_reg 2048 16 >> ./matrix_mul/blockmm_transpose_reg.csv 
! ./matrix_mul/blockmm_transpose_reg 2048 32 >> ./matrix_mul/blockmm_transpose_reg.csv 
! ./matrix_mul/blockmm_transpose_reg 2048 64 >> ./matrix_mul/blockmm_transpose_reg.csv
! ./matrix_mul/blockmm_transpose_reg 2048 128 >> ./matrix_mul/blockmm_transpose_reg.csv
! ./matrix_mul/blockmm_transpose_reg 2048 256 >> ./matrix_mul/blockmm_transpose_reg.csv

In [ ]:
display_df_mono(render_csv("matrix_mul/blockmm_transpose_reg.csv"))

In [ ]:
render_code("matrix_mul/rect_blockmm_trans.c", show=["//START","//END"])

In [ ]:
! cd matrix_mul; make rect_blockmm_trans; echo "size,tile_size_x,tile_size_y,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 8 8 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 8 16 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 16 8 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 16 16 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 32 8 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 32 16 >> ./matrix_mul/rect_blockmm_trans.csv 
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 64 8 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 128 8 >> ./matrix_mul/rect_blockmm_trans.csv
! taskset -c 8 ./matrix_mul/rect_blockmm_trans 2048 256 8 >> ./matrix_mul/rect_blockmm_trans.csv
display_df_mono(render_csv("matrix_mul/rect_blockmm_trans.csv"))

## Prefetch

x86 provide prefetch instructions. As a programmer, you may insert ```_mm_prefetch``` in x86 programs to perform software prefetch for your code. The gcc compiler also has a flag ```-fprefetch-loop-arrays``` to automatically insert software prefetch instructions.

### Using prefetch in matrix transpose code

The following example is a highly optimized matrix transpose code. In the example, we try to prefetch the next row.

In [ ]:
render_code("./prefetch/transpose.cpp", lang="c++", show=["//START", "//END"])

Now, let's take a look of what's happening!

In [ ]:
! cd prefetch; make clean; make
# ! echo "Without prefetch -- the baseline"; ssh htseng@celebi "lscpu | grep Model; cd courses/CS203/demo/memory/prefetch/; ./transpose"
! echo "Without prefetch -- the baseline"
! lscpu | grep Model
! ./prefetch/transpose
! echo "With prefetch"
! ./prefetch/transpose_prefetch

Let's try a different machine now.

In [ ]:
! ssh htseng@xerneas "cd /nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@xerneas  "/nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@xerneas  "/nfshome/htseng/courses/CSE142/demo/software_optimizations_memory/prefetch/transpose_prefetch"

In [ ]:
! ssh htseng@blissey "cd /nfshome/htseng/courses/CSE142/demo/memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@blissey  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@blissey  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose_prefetch"

In [ ]:
! ssh htseng@eevee "cd /nfshome/htseng/courses/CSE142/demo/memory/; make -C ./prefetch clean; make -C ./prefetch ; lscpu | grep Model"
! echo "Without prefetch -- the baseline"; ssh htseng@eevee  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose"
! echo "With prefetch";  ssh htseng@eevee  "/nfshome/htseng/courses/CSE142/demo/memory/prefetch/transpose_prefetch"


-- It doesn't work always!

In [ ]:
render_code("matrix_mul/blockmm_interchange.c", show=["//START","//END"])

In [ ]:
! cd matrix_mul; rm -f blockmm_interchange; make blockmm_interchange; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm_interchange.csv
! ./matrix_mul/blockmm_interchange 2048 8 >> ./matrix_mul/blockmm_interchange.csv 
! ./matrix_mul/blockmm_interchange 2048 16 >> ./matrix_mul/blockmm_interchange.csv 
! ./matrix_mul/blockmm_interchange 2048 32 >> ./matrix_mul/blockmm_interchange.csv 
! ./matrix_mul/blockmm_interchange 2048 64 >> ./matrix_mul/blockmm_interchange.csv
! ./matrix_mul/blockmm_interchange 2048 128 >> ./matrix_mul/blockmm_interchange.csv
! ./matrix_mul/blockmm_interchange 2048 256 >> ./matrix_mul/blockmm_interchange.csv
! cd matrix_mul; echo "size,tile_size,IC,Cycles,CPI,CT_ns,ET_s,DL1_miss_rate,DL1_misses,DL1_accesses" > blockmm.csv
! ./matrix_mul/blockmm 2048 16 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 32 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 64 >> ./matrix_mul/blockmm.csv
! ./matrix_mul/blockmm 2048 128 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 256 >> ./matrix_mul/blockmm.csv 
! ./matrix_mul/blockmm 2048 512 >> ./matrix_mul/blockmm.csv 
display_df_mono(render_csv("matrix_mul/blockmm.csv"))
display_df_mono(render_csv("matrix_mul/blockmm_interchange.csv"))
